# Notebook 04: Quantum Circuits — Deep Dive

---

**Author:** Milan Amrut Joshi  
**Date:** 2026-03-25  
**Prerequisites:** Notebooks 01-03  
**Framework:** Qiskit 1.x

---

## Table of Contents

1. The Circuit Model of Quantum Computing
2. Universal Gate Sets
3. Single-Qubit Gate Decompositions
4. Controlled Gates: CX, CZ, CCX, CSWAP
5. Multi-Controlled Gates
6. Circuit Identities and Equivalences
7. Circuit Depth, Width, and Complexity
8. Transpilation and Optimization
9. Building Complex Circuits
10. Exercises

## 1. The Circuit Model of Quantum Computing

The **quantum circuit model** is the most widely used model for quantum computation. It is the quantum analog of classical Boolean circuits.

### Components of a Quantum Circuit

```
                    ┌───┐          ┌───┐     ┌─┐
    |0⟩ ────────────┤ H ├────●─────┤ T ├─────┤M├──
                    └───┘    │     └───┘     └─┘
    |0⟩ ─────────────────────⊕───────────────┤M├──
                                              └─┘
    ↑                ↑       ↑       ↑        ↑
    Qubits          Gate   2-qubit  Gate   Measurement
    (wires)                 gate
```

1. **Qubits (wires):** Horizontal lines representing quantum bits, initialized to $|0\rangle$
2. **Quantum gates:** Boxes/symbols representing unitary operations
3. **Measurements:** Terminal operations that extract classical information
4. **Classical bits:** Double lines carrying measurement results
5. **Time flows left to right**

### Formal Definition

A quantum circuit on $n$ qubits is a sequence of quantum gates $U_1, U_2, \ldots, U_m$ applied to subsets of qubits. The overall unitary is:

$$U = U_m \cdot U_{m-1} \cdots U_2 \cdot U_1$$

Note: gates are applied **right to left** (like matrix multiplication), but drawn **left to right** in circuit diagrams.

### Circuit Properties

| Property | Description |
|----------|-------------|
| **Width** | Number of qubits (wires) |
| **Depth** | Maximum number of sequential gate layers |
| **Size** | Total number of gates |
| **T-count** | Number of T gates (important for fault tolerance) |
| **CNOT count** | Number of 2-qubit gates (often the bottleneck) |

## 2. Universal Gate Sets

A set of gates is **universal** if any unitary operation can be approximated to arbitrary precision using only gates from that set.

### Theorem (Solovay-Kitaev)

Any single-qubit unitary can be approximated to precision $\epsilon$ using $O(\log^c(1/\epsilon))$ gates from a finite universal set, where $c \approx 3.97$.

### Common Universal Gate Sets

| Gate Set | Gates | Notes |
|----------|-------|-------|
| Clifford + T | $\{H, S, T, \text{CNOT}\}$ | Most common for fault tolerance |
| Rotation + CNOT | $\{R_x, R_y, R_z, \text{CNOT}\}$ | Continuous rotations |
| IBM basis | $\{\sqrt{X}, R_z, \text{CNOT}\}$ | IBM hardware native |
| Google basis | $\{\sqrt{W}, R_z, \sqrt{\text{iSWAP}}\}$ | Google Sycamore native |

### Why Universality Matters

```
Any quantum algorithm
        │
        ▼
Abstract unitary U
        │
        ▼ (decompose)
Sequence of universal gates
        │
        ▼ (compile to hardware)
Native gate set of device
```

### The Clifford Group

The **Clifford group** is generated by $\{H, S, \text{CNOT}\}$. Clifford circuits alone are NOT universal — they can be efficiently simulated classically (Gottesman-Knill theorem). Adding the **T gate** makes the set universal.

$$\text{Clifford} \subset \text{Clifford + T} = \text{Universal}$$

In [ ]:
# Demonstrate that {H, T, CNOT} is universal by building arbitrary rotations
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

# H and T can approximate any single-qubit rotation
# HT = rotation, HTH = different rotation, etc.

# Build some composite gates from H and T
def get_unitary(circuit):
    return Operator(circuit).data

# Identity
qc_I = QuantumCircuit(1)

# Just H
qc_H = QuantumCircuit(1)
qc_H.h(0)

# Just T
qc_T = QuantumCircuit(1)
qc_T.t(0)

# HT
qc_HT = QuantumCircuit(1)
qc_HT.h(0)
qc_HT.t(0)

# HTH
qc_HTH = QuantumCircuit(1)
qc_HTH.h(0)
qc_HTH.t(0)
qc_HTH.h(0)

# HTHT
qc_HTHT = QuantumCircuit(1)
for _ in range(2):
    qc_HTHT.h(0)
    qc_HTHT.t(0)

print("=== Gate Compositions from {H, T} ===")
for name, qc in [('I', qc_I), ('H', qc_H), ('T', qc_T), 
                  ('HT', qc_HT), ('HTH', qc_HTH), ('HTHT', qc_HTHT)]:
    U = get_unitary(qc)
    print(f"\n{name}:")
    print(np.round(U, 4))

print("\nEach composition gives a DIFFERENT unitary — by combining enough")
print("H and T gates, we can approximate any single-qubit unitary!")

## 3. Single-Qubit Gate Decompositions

### ZYZ Decomposition

Any single-qubit unitary $U$ can be decomposed as:

$$U = e^{i\alpha} R_z(\beta) R_y(\gamma) R_z(\delta)$$

where:
- $\alpha$ is a global phase (physically irrelevant for single qubits)
- $R_z(\theta) = e^{-i\theta Z/2} = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$
- $R_y(\theta) = e^{-i\theta Y/2} = \begin{pmatrix} \cos\theta/2 & -\sin\theta/2 \\ \sin\theta/2 & \cos\theta/2 \end{pmatrix}$

### ZXZ Decomposition

$$U = e^{i\alpha} R_z(\beta) R_x(\gamma) R_z(\delta)$$

### U3 Gate (Qiskit)

Qiskit's general single-qubit gate:

$$U_3(\theta, \phi, \lambda) = \begin{pmatrix} \cos\frac{\theta}{2} & -e^{i\lambda}\sin\frac{\theta}{2} \\ e^{i\phi}\sin\frac{\theta}{2} & e^{i(\phi+\lambda)}\cos\frac{\theta}{2} \end{pmatrix}$$

This can represent ANY single-qubit gate with just 3 parameters.

### Common Gates as U3

| Gate | U3 Parameters | Expression |
|------|--------------|------------|
| X | $U_3(\pi, 0, \pi)$ | 180° rotation around X |
| Y | $U_3(\pi, \pi/2, \pi/2)$ | 180° rotation around Y |
| Z | $U_3(0, 0, \pi)$ = $R_z(\pi)$ | 180° rotation around Z |
| H | $U_3(\pi/2, 0, \pi)$ | Hadamard |
| S | $U_3(0, 0, \pi/2)$ = $R_z(\pi/2)$ | Phase gate |
| T | $U_3(0, 0, \pi/4)$ = $R_z(\pi/4)$ | T gate |

In [ ]:
# Demonstrate gate decompositions
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import numpy as np

# Verify U3 representations of common gates
gates_u3 = {
    'X': (np.pi, 0, np.pi),
    'Y': (np.pi, np.pi/2, np.pi/2),
    'H': (np.pi/2, 0, np.pi),
    'S': (0, 0, np.pi/2),
    'T': (0, 0, np.pi/4),
}

# Reference gates
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)

reference = {'X': X, 'Y': Y, 'H': H, 'S': S, 'T': T}

print("=== U3 Decomposition Verification ===")
for name, (theta, phi, lam) in gates_u3.items():
    # Build U3 manually
    u3 = np.array([
        [np.cos(theta/2), -np.exp(1j*lam)*np.sin(theta/2)],
        [np.exp(1j*phi)*np.sin(theta/2), np.exp(1j*(phi+lam))*np.cos(theta/2)]
    ])
    
    # Check if equal up to global phase
    ref = reference[name]
    # Find global phase
    nonzero_idx = np.argmax(np.abs(ref))
    phase = u3.flat[nonzero_idx] / ref.flat[nonzero_idx]
    match = np.allclose(u3, phase * ref)
    
    print(f"  {name} = U3({theta/np.pi:.2f}π, {phi/np.pi:.2f}π, {lam/np.pi:.2f}π): {'✓' if match else '✗'}")

# Demonstrate ZYZ decomposition
print("\n=== ZYZ Decomposition Example ===")
# Decompose H = Rz(β) Ry(γ) Rz(δ)
# H = Rz(π) Ry(π/2) Rz(0) up to global phase
def Rz(theta):
    return np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])

def Ry(theta):
    return np.array([[np.cos(theta/2), -np.sin(theta/2)], 
                     [np.sin(theta/2), np.cos(theta/2)]])

# H ∝ Rz(π) Ry(π/2) Rz(0)
decomposed_H = Rz(np.pi) @ Ry(np.pi/2)
print(f"Rz(π)·Ry(π/2) =\n{np.round(decomposed_H, 4)}")
print(f"H =\n{np.round(H, 4)}")
# Check up to global phase
phase = decomposed_H[0,0] / H[0,0]
print(f"Global phase factor: {phase:.4f}")
print(f"Match (up to phase): {np.allclose(decomposed_H, phase * H)}")

## 4. Controlled Gates: CX, CZ, CCX, CSWAP

### Controlled-X (CNOT / CX)

$$\text{CX} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes X = \begin{pmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&0&1 \\ 0&0&1&0 \end{pmatrix}$$

```
Control: ───●───      Flips target iff control = |1⟩
            │
Target:  ───⊕───
```

### Controlled-Z (CZ)

$$\text{CZ} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes Z = \begin{pmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&1&0 \\ 0&0&0&-1 \end{pmatrix}$$

```
Qubit 1: ───●───      Adds phase -1 to |11⟩
            │          CZ is symmetric! Control/target are interchangeable.
Qubit 2: ───●───
```

**Key identity:** CZ = (I ⊗ H) · CX · (I ⊗ H)

### Toffoli Gate (CCX / CCNOT)

The **Toffoli gate** is a 3-qubit gate with 2 controls and 1 target:

$$\text{CCX}|a,b,c\rangle = |a,b, c \oplus (a \cdot b)\rangle$$

It flips the target only when **both** controls are $|1\rangle$.

```
Control 1: ───●───
              │
Control 2: ───●───
              │
Target:    ───⊕───
```

The Toffoli gate is **universal for classical computation** — any Boolean function can be built from Toffoli gates.

### Fredkin Gate (CSWAP)

The **Fredkin gate** swaps two qubits conditioned on a control qubit:

$$\text{CSWAP}|a,b,c\rangle = \begin{cases} |a,b,c\rangle & \text{if } a=0 \\ |a,c,b\rangle & \text{if } a=1 \end{cases}$$

```
Control: ───●───
            │
Qubit 1: ───×───
            │
Qubit 2: ───×───
```

In [ ]:
# Demonstrate all controlled gates
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
import numpy as np

# CX (CNOT)
print("=== CX (CNOT) Gate ===")
qc_cx = QuantumCircuit(2)
qc_cx.cx(0, 1)
print(qc_cx.draw('text'))
print(f"Matrix:\n{np.round(Operator(qc_cx).data.real, 0)}")

# CZ
print("\n=== CZ Gate ===")
qc_cz = QuantumCircuit(2)
qc_cz.cz(0, 1)
print(qc_cz.draw('text'))
print(f"Matrix:\n{np.round(Operator(qc_cz).data.real, 0)}")

# Verify CZ = (I⊗H)·CX·(I⊗H)
qc_cz_from_cx = QuantumCircuit(2)
qc_cz_from_cx.h(1)
qc_cz_from_cx.cx(0, 1)
qc_cz_from_cx.h(1)
print(f"\nCZ = H·CX·H? {np.allclose(Operator(qc_cz).data, Operator(qc_cz_from_cx).data)}")

# CCX (Toffoli)
print("\n=== CCX (Toffoli) Gate ===")
qc_ccx = QuantumCircuit(3)
qc_ccx.ccx(0, 1, 2)
print(qc_ccx.draw('text'))

# Verify truth table
print("\nToffoli Truth Table:")
for a in range(2):
    for b in range(2):
        for c in range(2):
            qc = QuantumCircuit(3)
            if a: qc.x(0)
            if b: qc.x(1)
            if c: qc.x(2)
            qc.ccx(0, 1, 2)
            sv = Statevector.from_instruction(qc)
            outcome = max(sv.probabilities_dict(), key=sv.probabilities_dict().get)
            print(f"  |{a}{b}{c}⟩ → |{outcome}⟩")

In [ ]:
# CSWAP (Fredkin) Gate
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print("=== CSWAP (Fredkin) Gate ===")
qc_cswap = QuantumCircuit(3)
qc_cswap.cswap(0, 1, 2)
print(qc_cswap.draw('text'))

print("\nFredkin Truth Table:")
for a in range(2):
    for b in range(2):
        for c in range(2):
            qc = QuantumCircuit(3)
            if a: qc.x(0)
            if b: qc.x(1)
            if c: qc.x(2)
            qc.cswap(0, 1, 2)
            sv = Statevector.from_instruction(qc)
            outcome = max(sv.probabilities_dict(), key=sv.probabilities_dict().get)
            swapped = "swapped" if a == 1 and b != c else "no swap"
            print(f"  |{a}{b}{c}⟩ → |{outcome}⟩  ({swapped})")

## 5. Multi-Controlled Gates

### General Controlled-U

A **controlled-U** gate applies unitary $U$ to the target qubit(s) only when the control qubit is $|1\rangle$:

$$C(U) = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes U$$

### Multi-Controlled-X (MCX)

An $n$-controlled X gate flips the target only when ALL $n$ control qubits are $|1\rangle$:

```
C₁: ───●───
       │
C₂: ───●───       Flips target iff C₁=C₂=...=Cₙ=|1⟩
       │
  ⋮    ⋮
       │
Cₙ: ───●───
       │
T:  ───⊕───
```

### Decomposition of Toffoli

The Toffoli gate can be decomposed into 1-qubit and CNOT gates:

$$\text{CCX} = \text{(6 CNOTs + several single-qubit gates)}$$

This decomposition is important because real quantum hardware typically only supports 1-qubit and 2-qubit gates.

In [ ]:
# Multi-controlled gates and their decompositions
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit import transpile
import numpy as np

# Controlled-H
print("=== Controlled-H Gate ===")
qc_ch = QuantumCircuit(2)
qc_ch.ch(0, 1)
print(qc_ch.draw('text'))
print(f"\nMatrix (real part):\n{np.round(Operator(qc_ch).data.real, 3)}")

# Controlled-S
print("\n=== Controlled-S (CPhase π/2) Gate ===")
qc_cs = QuantumCircuit(2)
qc_cs.cs(0, 1)
print(qc_cs.draw('text'))

# Multi-controlled X (3 controls)
print("\n=== 3-Controlled X (C³X) Gate ===")
qc_c3x = QuantumCircuit(4)
qc_c3x.mcx([0, 1, 2], 3)  # 3 controls, 1 target
print(qc_c3x.draw('text'))

# Verify: only flips when all controls are 1
print("\nC³X verification (only showing non-trivial cases):")
for bits in ['0111', '1011', '1101', '1110', '1111']:
    qc = QuantumCircuit(4)
    for i, b in enumerate(reversed(bits)):
        if b == '1': qc.x(i)
    qc.mcx([0, 1, 2], 3)
    sv = Statevector.from_instruction(qc)
    outcome = max(sv.probabilities_dict(), key=sv.probabilities_dict().get)
    print(f"  |{bits}⟩ → |{outcome}⟩")

## 6. Circuit Identities and Equivalences

Understanding circuit identities is crucial for optimization. Here are the most important ones:

### Identity 1: Gate Inverses

$$HH = I, \quad XX = I, \quad YY = I, \quad ZZ = I$$
$$SS^\dagger = I, \quad TT^\dagger = I, \quad \text{CNOT} \cdot \text{CNOT} = I$$

### Identity 2: CNOT Conjugations

$$\text{CX} \cdot (I \otimes Z) \cdot \text{CX} = I \otimes Z$$
$$\text{CX} \cdot (Z \otimes I) \cdot \text{CX} = Z \otimes Z$$
$$\text{CX} \cdot (I \otimes X) \cdot \text{CX} = X \otimes X$$

### Identity 3: CZ from CX

$$\text{CZ} = (I \otimes H) \cdot \text{CX} \cdot (I \otimes H)$$

### Identity 4: SWAP from CNOTs

$$\text{SWAP} = \text{CX}_{01} \cdot \text{CX}_{10} \cdot \text{CX}_{01}$$

```
───●───⊕───●───     ───×───
   │   │   │     =     │
───⊕───●───⊕───     ───×───
```

### Identity 5: Controlled-Z Symmetry

$$\text{CZ}_{01} = \text{CZ}_{10}$$

CZ is symmetric — it doesn't matter which qubit is the control!

### Identity 6: Pauli Commutation with CNOT

```
───●───X───  =  ───X───●───
   │                   │
───⊕───────     ───X───⊕───

───●───────  =  ───Z───●───
   │                   │
───⊕───Z───     ───────⊕───
```

In [ ]:
# Verify circuit identities
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import numpy as np

def circuits_equivalent(qc1, qc2):
    """Check if two circuits implement the same unitary (up to global phase)."""
    U1 = Operator(qc1).data
    U2 = Operator(qc2).data
    # Check if U1 = e^(iφ) U2 for some φ
    nonzero = np.argmax(np.abs(U1))
    if np.abs(U1.flat[nonzero]) < 1e-10:
        return np.allclose(U1, U2, atol=1e-8)
    phase = U1.flat[nonzero] / U2.flat[nonzero]
    return np.allclose(U1, phase * U2, atol=1e-8)

# Identity 1: SWAP = CX·CX·CX
print("=== Circuit Identity Verification ===")

qc_swap = QuantumCircuit(2)
qc_swap.swap(0, 1)

qc_swap_from_cx = QuantumCircuit(2)
qc_swap_from_cx.cx(0, 1)
qc_swap_from_cx.cx(1, 0)
qc_swap_from_cx.cx(0, 1)

print(f"\n1. SWAP = CX₀₁·CX₁₀·CX₀₁: {circuits_equivalent(qc_swap, qc_swap_from_cx)}")

# Identity 2: CZ = H·CX·H
qc_cz = QuantumCircuit(2)
qc_cz.cz(0, 1)

qc_cz_from_cx = QuantumCircuit(2)
qc_cz_from_cx.h(1)
qc_cz_from_cx.cx(0, 1)
qc_cz_from_cx.h(1)

print(f"2. CZ = (I⊗H)·CX·(I⊗H): {circuits_equivalent(qc_cz, qc_cz_from_cx)}")

# Identity 3: CNOT is self-inverse
qc_cx2 = QuantumCircuit(2)
qc_cx2.cx(0, 1)
qc_cx2.cx(0, 1)

qc_id = QuantumCircuit(2)  # Identity

print(f"3. CX·CX = I: {circuits_equivalent(qc_cx2, qc_id)}")

# Identity 4: CZ is symmetric
qc_cz01 = QuantumCircuit(2)
qc_cz01.cz(0, 1)

qc_cz10 = QuantumCircuit(2)
qc_cz10.cz(1, 0)

print(f"4. CZ₀₁ = CZ₁₀ (symmetric): {circuits_equivalent(qc_cz01, qc_cz10)}")

# Identity 5: H·Z·H = X
qc_hzh = QuantumCircuit(1)
qc_hzh.h(0)
qc_hzh.z(0)
qc_hzh.h(0)

qc_x = QuantumCircuit(1)
qc_x.x(0)

print(f"5. HZH = X: {circuits_equivalent(qc_hzh, qc_x)}")

# Identity 6: H·X·H = Z
qc_hxh = QuantumCircuit(1)
qc_hxh.h(0)
qc_hxh.x(0)
qc_hxh.h(0)

qc_z = QuantumCircuit(1)
qc_z.z(0)

print(f"6. HXH = Z: {circuits_equivalent(qc_hxh, qc_z)}")

## 7. Circuit Depth, Width, and Complexity

### Definitions

- **Width:** Number of qubits = number of wires
- **Depth:** Length of the longest path from input to output (number of gate layers that must be executed sequentially)
- **Size:** Total number of gates
- **T-depth:** Number of layers containing T gates
- **CNOT count:** Total number of CNOT (2-qubit) gates

### Why Depth Matters

On real quantum hardware:
- Each gate introduces **errors** (gate infidelity)
- Qubits have limited **coherence time** (they "forget" their state)
- Deeper circuits → more errors → less reliable results

```
Depth 3 circuit:                     Depth 2 circuit (optimized):

Layer1  Layer2  Layer3               Layer1    Layer2
──H──────●──────T───                 ──H──T─────●────
         │                                      │
─────────⊕──────H───                 ──H────────⊕────

Same result, but fewer layers = less noise!
```

### Parallelism

Gates on **different qubits** can be executed simultaneously (in the same layer). This is why depth can be less than size.

In [ ]:
# Analyze circuit metrics
from qiskit import QuantumCircuit

def analyze_circuit(qc, name="Circuit"):
    """Print circuit metrics."""
    print(f"\n=== {name} ===")
    print(qc.draw('text'))
    print(f"  Width (qubits): {qc.num_qubits}")
    print(f"  Depth: {qc.depth()}")
    print(f"  Size (total gates): {qc.size()}")
    
    # Count gate types
    ops = qc.count_ops()
    print(f"  Gate counts: {dict(ops)}")
    return qc.depth(), qc.size()

# Example 1: Simple circuit
qc1 = QuantumCircuit(3)
qc1.h(0)
qc1.cx(0, 1)
qc1.cx(1, 2)
analyze_circuit(qc1, "GHZ State Creation")

# Example 2: Parallel gates
qc2 = QuantumCircuit(4)
qc2.h(0)  # These can run in parallel
qc2.h(1)  # ^
qc2.h(2)  # ^
qc2.h(3)  # ^
qc2.cx(0, 1)  # These can run in parallel
qc2.cx(2, 3)  # ^
analyze_circuit(qc2, "Parallel Gates")

# Example 3: Deep circuit
qc3 = QuantumCircuit(2)
for _ in range(5):
    qc3.h(0)
    qc3.cx(0, 1)
    qc3.t(1)
analyze_circuit(qc3, "Deep Circuit (5 repetitions)")

## 8. Transpilation and Optimization

**Transpilation** converts a quantum circuit into an equivalent circuit that uses only the **native gates** of a target hardware device, while **optimizing** for depth and gate count.

### Transpilation Steps

```
Original Circuit
       │
       ▼
1. Unroll to basis gates (e.g., {CX, Rz, SX, X})
       │
       ▼
2. Map to hardware topology (qubit connectivity)
       │
       ▼
3. Route: insert SWAPs for non-adjacent qubits
       │
       ▼
4. Optimize: cancel gates, merge rotations
       │
       ▼
Optimized Hardware-Compatible Circuit
```

### Optimization Levels in Qiskit

| Level | Description | Speed | Quality |
|-------|-------------|-------|---------|
| 0 | No optimization | Fastest | Worst |
| 1 | Light optimization | Fast | Good |
| 2 | Medium optimization | Medium | Better |
| 3 | Heavy optimization | Slowest | Best |

In [ ]:
# Demonstrate transpilation and optimization
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
import numpy as np

# Create a circuit with high-level gates
qc = QuantumCircuit(3)
qc.h(0)
qc.ccx(0, 1, 2)  # Toffoli — not a native gate on most hardware
qc.swap(0, 2)

print("=== Original Circuit ===")
print(qc.draw('text'))
print(f"Depth: {qc.depth()}, Size: {qc.size()}, Gates: {dict(qc.count_ops())}")

# Transpile to a basic gate set
basis_gates = ['cx', 'rz', 'sx', 'x']  # IBM native gates

for opt_level in [0, 1, 2, 3]:
    transpiled = transpile(qc, basis_gates=basis_gates, optimization_level=opt_level)
    print(f"\n=== Optimization Level {opt_level} ===")
    print(f"Depth: {transpiled.depth()}, Size: {transpiled.size()}, Gates: {dict(transpiled.count_ops())}")
    cx_count = transpiled.count_ops().get('cx', 0)
    print(f"CNOT count: {cx_count}")

# Verify equivalence
transpiled_3 = transpile(qc, basis_gates=basis_gates, optimization_level=3)
print(f"\nOriginal and optimized circuits equivalent: {np.allclose(Operator(qc).data, Operator(transpiled_3).data)}")

In [ ]:
# Show the transpiled Toffoli gate decomposition
from qiskit import QuantumCircuit, transpile

# Just Toffoli
qc_toffoli = QuantumCircuit(3)
qc_toffoli.ccx(0, 1, 2)

# Transpile to basic gates
transpiled = transpile(qc_toffoli, basis_gates=['cx', 'rz', 'sx', 'x'], optimization_level=3)

print("=== Toffoli Gate Decomposition ===")
print("\nOriginal:")
print(qc_toffoli.draw('text'))
print(f"\nDecomposed into native gates (optimization level 3):")
print(transpiled.draw('text'))
print(f"\nGate counts: {dict(transpiled.count_ops())}")
print(f"CNOT count: {transpiled.count_ops().get('cx', 0)}")
print(f"Depth: {transpiled.depth()}")

## 9. Building Complex Circuits

Let's build some more complex quantum circuits to demonstrate the concepts.

In [ ]:
# Build a Quantum Fourier Transform (QFT) circuit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
import numpy as np

def qft_circuit(n):
    """Build an n-qubit Quantum Fourier Transform circuit."""
    qc = QuantumCircuit(n, name=f'QFT({n})')
    
    for i in range(n):
        qc.h(i)
        for j in range(i+1, n):
            angle = np.pi / (2 ** (j - i))
            qc.cp(angle, j, i)  # Controlled phase
    
    # Swap qubits to match convention
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    
    return qc

# Build 3-qubit QFT
qft3 = qft_circuit(3)
print("=== 3-Qubit Quantum Fourier Transform ===")
print(qft3.draw('text'))
print(f"\nDepth: {qft3.depth()}")
print(f"Size: {qft3.size()}")
print(f"Gates: {dict(qft3.count_ops())}")

# Verify: QFT matrix should be the DFT matrix (up to normalization)
n = 3
N = 2**n
omega = np.exp(2j * np.pi / N)
DFT = np.array([[omega**(j*k) for k in range(N)] for j in range(N)]) / np.sqrt(N)

QFT_matrix = Operator(qft3).data
print(f"\nQFT matches DFT matrix: {np.allclose(np.abs(QFT_matrix), np.abs(DFT), atol=1e-6)}")

In [ ]:
# Build a Grover's oracle and diffusion operator
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def grover_oracle(n, target):
    """Build an oracle that marks the target state with a phase flip."""
    qc = QuantumCircuit(n, name=f'Oracle({target})')
    
    # Flip qubits where target bit is 0
    target_bits = format(target, f'0{n}b')
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)
    
    # Multi-controlled Z (implemented as H-MCX-H on last qubit)
    if n == 2:
        qc.cz(0, 1)
    elif n == 3:
        qc.h(n-1)
        qc.ccx(0, 1, n-1)
        qc.h(n-1)
    
    # Undo the X flips
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)
    
    return qc

def grover_diffusion(n):
    """Build the Grover diffusion operator."""
    qc = QuantumCircuit(n, name='Diffusion')
    
    # H on all qubits
    for i in range(n):
        qc.h(i)
    
    # Phase flip |0...0⟩
    for i in range(n):
        qc.x(i)
    
    if n == 2:
        qc.cz(0, 1)
    elif n == 3:
        qc.h(n-1)
        qc.ccx(0, 1, n-1)
        qc.h(n-1)
    
    for i in range(n):
        qc.x(i)
    for i in range(n):
        qc.h(i)
    
    return qc

# Build 2-qubit Grover's algorithm searching for |11⟩
n = 2
target = 3  # |11⟩

grover = QuantumCircuit(n, n)
# Initial superposition
for i in range(n):
    grover.h(i)
grover.barrier()

# Oracle + Diffusion (1 iteration suffices for n=2)
grover.compose(grover_oracle(n, target), inplace=True)
grover.barrier()
grover.compose(grover_diffusion(n), inplace=True)
grover.barrier()

# Measure
grover.measure(range(n), range(n))

print("=== Grover's Algorithm (2 qubits, target = |11⟩) ===")
print(grover.draw('text'))

# Simulate
from qiskit_aer import AerSimulator
simulator = AerSimulator()
result = simulator.run(grover, shots=1000).result()
counts = result.get_counts()
print(f"\nResults: {counts}")
print(f"Target |{format(target, f'0{n}b')}⟩ found with probability: {counts.get(format(target, f'0{n}b'), 0)/1000:.1%}")

In [ ]:
# Build a quantum adder circuit (simple 1-bit + 1-bit = 2-bit)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

def quantum_half_adder():
    """1-bit + 1-bit quantum half adder.
    Input: |a⟩|b⟩|0⟩ (carry bit initialized to 0)
    Output: |a⟩|a⊕b⟩|a·b⟩ (a unchanged, sum=a⊕b, carry=a·b)
    """
    qc = QuantumCircuit(3, 2, name='Half Adder')
    # Carry: AND gate = Toffoli
    qc.ccx(0, 1, 2)  # carry = a AND b
    # Sum: XOR gate = CNOT
    qc.cx(0, 1)       # sum = a XOR b
    return qc

print("=== Quantum Half Adder ===")

simulator = AerSimulator()

# Test all input combinations
print(f"{'Input (a,b)':<15} {'Sum':<6} {'Carry':<6} {'Decimal':<8}")
print("-" * 40)

for a in range(2):
    for b in range(2):
        qc = QuantumCircuit(3, 2)
        if a: qc.x(0)
        if b: qc.x(1)
        # carry initialized to |0⟩
        qc.ccx(0, 1, 2)  # carry
        qc.cx(0, 1)       # sum
        qc.measure(1, 0)  # measure sum
        qc.measure(2, 1)  # measure carry
        
        result = simulator.run(qc, shots=100).result()
        outcome = list(result.get_counts().keys())[0]
        carry, sumbit = int(outcome[0]), int(outcome[1])
        print(f"  {a} + {b:<10} {sumbit:<6} {carry:<6} = {a+b}")

print("\nThe quantum half adder correctly computes binary addition!")

## 10. Exercises

### Exercise 1: SWAP Test
The **SWAP test** determines if two quantum states are equal without measuring them. Implement it:
1. Prepare an ancilla in $|0\rangle$, apply H
2. Apply CSWAP (controlled by ancilla) on the two test states
3. Apply H to ancilla, measure

If the states are identical, the ancilla always gives $|0\rangle$.

### Exercise 2: Build QFT for 4 Qubits
Extend the QFT circuit to 4 qubits. Analyze its depth, CNOT count, and verify it produces the correct DFT matrix.

### Exercise 3: Transpile to Google Basis
Transpile the Bell state circuit to Google's native gate set $\{\sqrt{\text{iSWAP}}, R_z, \sqrt{W}\}$. Compare the gate count with IBM's basis.

### Exercise 4: Circuit Optimization
Create a deliberately inefficient circuit (e.g., with redundant gate pairs HH, XX) and show that transpilation removes them.

### Exercise 5: Build a Quantum Comparator
Build a circuit that takes two single-qubit inputs $|a\rangle$ and $|b\rangle$ and outputs $|a > b\rangle$ on an ancilla qubit.

In [ ]:
# Exercise 4 Solution: Optimize a redundant circuit
from qiskit import QuantumCircuit, transpile

# Deliberately inefficient circuit
qc_bad = QuantumCircuit(2)
qc_bad.h(0)
qc_bad.h(0)   # H·H = I (redundant!)
qc_bad.x(1)
qc_bad.x(1)   # X·X = I (redundant!)
qc_bad.cx(0, 1)
qc_bad.cx(0, 1)  # CX·CX = I (redundant!)
qc_bad.h(0)   # This is the only gate that matters
qc_bad.cx(0, 1)  # And this one

print("=== Redundant Circuit ===")
print(qc_bad.draw('text'))
print(f"Depth: {qc_bad.depth()}, Size: {qc_bad.size()}")

# Optimize
qc_opt = transpile(qc_bad, basis_gates=['cx', 'rz', 'sx', 'x'], optimization_level=3)

print("\n=== Optimized Circuit ===")
print(qc_opt.draw('text'))
print(f"Depth: {qc_opt.depth()}, Size: {qc_opt.size()}")
print(f"\nReduction: {qc_bad.size()} → {qc_opt.size()} gates")

# Verify equivalence
from qiskit.quantum_info import Operator
import numpy as np
print(f"Circuits equivalent: {np.allclose(Operator(qc_bad).data, Operator(qc_opt).data)}")

## Summary

In this notebook, we covered:

1. **Circuit model** — qubits, gates, measurements as the building blocks of quantum computation
2. **Universal gate sets** — $\{H, T, \text{CNOT}\}$ can approximate any unitary operation
3. **Gate decompositions** — ZYZ, U3 parametrizations for single-qubit gates
4. **Controlled gates** — CX, CZ, Toffoli (CCX), Fredkin (CSWAP) with matrix representations
5. **Circuit identities** — SWAP from CNOTs, CZ from CX+H, HZH=X, and more
6. **Circuit complexity** — depth, width, size, CNOT count
7. **Transpilation** — converting abstract circuits to hardware-native gates with optimization
8. **Complex circuits** — QFT, Grover's oracle, quantum adder

### Key Takeaway

Real quantum circuits must be compiled (transpiled) to the native gate set of the target hardware. Circuit optimization is crucial because fewer gates and lower depth mean less noise and better results.

### Next Notebook

In **Notebook 05**, we implement the **quantum teleportation protocol** — a beautiful application of entanglement and classical communication.